[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/specular_caustics/specular_caustics.ipynb)

# Specular Caustics

This example computes moving-water refractive caustics on a flat bottom, traces their singular fold curves from the ray map's Jacobian determinant, and solves for multiple specular light paths reaching a single receiver point, following the constraint formulation of [Zeltner, Georgiev, and Jakob (2020)](https://rgl.epfl.ch/publications/Zeltner2020Specular). Parallel sunlight crossing a rippled water surface bends toward wave crests and spreads apart beneath troughs, folding the refracted ray sheet over itself so bright caustic networks sweep across the pool floor.

In three-dimensional geometric algebra, spatial displacements and ray directions are modelled as intrinsic antivectors while ripple wavevectors and tangent planes are extrinsic vectors. Pairing a horizontal wavevector with a surface entry point via `wavevector & position` evaluates the ripple phase, subtracting the slope 1-form from `z` defines the local interface plane `z - slope_gradient` for Snell refraction, and differentiating the horizontal landing map via `derivative(landing)` produces an intrinsic `Horizontal <- Horizontal` extensor whose `.det()` vanishes along caustic folds and whose `.solve(...)` pulls receiver residuals back into Newton steps on the water surface.

In the vector and coordinate-Jacobian notation of geometric optics, a height field $z = h(x, y)$ defines the unnormalized normal $\mathbf{n} = (-\partial_x h, -\partial_y h, 1) / \sqrt{1 + |\nabla h|^2}$, Snell's law writes the refracted unit vector as $\mathbf{t} = \eta\,\mathbf{i} + \bigl(\eta(-\mathbf{n}\cdot\mathbf{i}) - \sqrt{1 - \eta^2(1 - (\mathbf{n}\cdot\mathbf{i})^2)}\bigr)\mathbf{n}$, and locating caustics or specular paths requires assembling the $2 \times 2$ coordinate Jacobian $\partial(x_r, y_r)/\partial(x_s, y_s)$ through chain-rule derivatives of $\mathbf{n}(x_s, y_s)$ and the intersection distance $d = (-H - h)/t_z$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[jax,examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from numga import Algebra
from numga.algebras import VGA3D as ga
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import save_figure, show_states
from examples.optics.specular_caustics import render, scenarios

# Euclidean 3D space `x+y+z+`, with `+` extrinsic and `-` intrinsic.
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                       # Scalar+
Direction = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                        # Vector+
Wavevector = Plane.derive.within(Algebra("x+y+"))                                 # Vector+
Horizontal = Wavevector.derive.dual                                               # Antivector-

x, y = mv.x.dual(), mv.y.dual()                                                   # [] Direction-, Direction-
up = mv.z.dual()                                                                  # [] Direction-

## Refraction through travelling surface waves

Collimated vertical light enters a superposition of four travelling ripples, refracts across the normalized level-set plane `(z - slope_gradient).normalized()` at each entry point, and lands on a horizontal receiver below the surface. Tracing and binning every frame takes about ten seconds.

In [ ]:
amplitudes = scenarios.amplitudes
phases = scenarios.phases
phase_rates = scenarios.phase_rates
index_ratio = scenarios.index_ratio
depth = scenarios.depth
source_halfwidth = scenarios.source_halfwidth
receiver_halfwidth = scenarios.receiver_halfwidth
n_rays = scenarios.n_rays
n_frames = scenarios.n_frames
n_pixels = scenarios.n_pixels
blur_pixels = scenarios.blur_pixels
exposure = scenarios.exposure
frame_duration_ms = scenarios.frame_duration_ms
n_colours = scenarios.n_colours

wave_rotors = (-mv.xy * scenarios.wave_angles / 2).exp()                          # [4] Rotor+
wavevectors = (wave_rotors >> mv.x) * scenarios.wavenumbers                       # [4] Wavevector+
spacing = 2 * source_halfwidth / n_rays
centres = (np.arange(n_rays) + 0.5) * spacing - source_halfwidth
rng = np.random.default_rng(scenarios.seed)
# Stratified jitter inside each grid cell suppresses lattice aliasing in the binned irradiance.
x_offsets = rng.uniform(-0.5, 0.5, (n_rays, n_rays)) * spacing
y_offsets = rng.uniform(-0.5, 0.5, (n_rays, n_rays)) * spacing
positions = x * (centres[None, :] + x_offsets) + y * (centres[:, None] + y_offsets)  # [n_rays, n_rays] Horizontal-
ray_area = spacing ** 2

In [ ]:
def refracted_rays(
    position: Horizontal, wavevectors: Wavevector, amplitudes: np.ndarray,
    phases: np.ndarray, index_ratio: float,
) -> tuple[Direction, Plane, Direction]:
    """Vertical illumination hitting the rippled surface at horizontal `position` (`[...] Horizontal-`)
    refracts across unit surface planes (`[...] Plane+`) from surface points (`[...] Direction-`) into
    transmitted unit directions (`[...] Direction-`)."""
    wave_phases = (wavevectors & position[..., None]) - phases                    # [..., n_waves] Scalar+
    height = (amplitudes * wave_phases.cos()).sum(axis=-1)                        # [...] Scalar+
    slope_gradient = -(wavevectors * (amplitudes * wave_phases.sin())).sum(axis=-1)  # [...] Wavevector+
    # Level-set 1-form of `z - height == 0` normalized to unit length, and its dual upward normal.
    surface_plane = (mv.z - slope_gradient).normalized()                          # [...] Plane+
    surface_normal = surface_plane.dual()                                         # [...] Direction-

    # Snell's law scales the component tangential to the interface, and unit norm fixes the downward normal component.
    incoming = -up                                                                # [] Direction-
    tangent_part = index_ratio * (incoming - surface_normal * (surface_plane & incoming))  # [...] Direction-
    outgoing = tangent_part - surface_normal * (1 - tangent_part.scalar_norm_squared()).square_root()  # [...] Direction-
    surface_point = position + up * height                                        # [...] Direction-
    return surface_point, surface_plane, outgoing


def receiver_landing(
    position: Horizontal, wavevectors: Wavevector, amplitudes: np.ndarray,
    phases: np.ndarray, depth: float, index_ratio: float,
) -> Horizontal:
    """Tracing refracted rays from horizontal `position` (`[...] Horizontal-`) down to `z = -depth`
    yields their horizontal landing positions (`[...] Horizontal-`) on the receiver plane."""
    surface_point, _, outgoing = refracted_rays(position, wavevectors, amplitudes, phases, index_ratio)
    ray_distance = (-depth - (mv.z & surface_point)) / (mv.z & outgoing)          # [...] Scalar+
    landing_point = surface_point + ray_distance * outgoing                       # [...] Direction-
    return landing_point.cast(Horizontal)                                         # [...] Horizontal-

In [ ]:
def caustic_states(positions: Horizontal, wavevectors: Wavevector, n_steps: int) -> Iterator[Horizontal]:
    """Yields `[n_rays, n_rays] Horizontal-` receiver landings across `n_steps` frames of one wave period."""
    for time in np.linspace(0, 2 * np.pi, n_steps, endpoint=False):
        yield receiver_landing(positions, wavevectors, amplitudes, phases + phase_rates * time, depth, index_ratio)


landing_frames = show_states(
    caustic_states(positions, wavevectors, n_frames), render.caustics_animation,
    receiver_halfwidth, ray_area, n_pixels, blur_pixels, exposure,
    name="specular_caustics", duration_ms=frame_duration_ms, n_frames=n_frames, colors=n_colours,
)

Warm bright ridges trace the caustics across the dark blue receiver as the four ripple trains move past one another, splitting and rejoining wherever wave crests focus incoming light onto the bottom.

## Singular fold curves of the landing map

Along curves on the water where an infinitesimal surface patch collapses onto a line at the receiver, the horizontal Jacobian `derivative(landing)(chart)` loses rank and its determinant crosses zero. Mapping those zero-determinant contours forward to the receiver places them directly along the bright caustic ridges.

In [ ]:
n_derivative = scenarios.n_derivative
chart_samples = np.linspace(-source_halfwidth, source_halfwidth, n_derivative)
chart = x * chart_samples[None, :] + y * chart_samples[:, None]                   # [n_derivative, n_derivative] Horizontal-

In [ ]:
landing = partial(receiver_landing, wavevectors=wavevectors, amplitudes=amplitudes, phases=phases,
                  depth=depth, index_ratio=index_ratio)
local_jacobian = derivative(landing)(chart)                                       # [n_derivative, n_derivative] Horizontal- <- Horizontal-
area_ratios = local_jacobian.det()                                                # [n_derivative, n_derivative] Scalar+
folds_fig = render.folds_figure(landing_frames[0], landing(chart), area_ratios, receiver_halfwidth,
                                ray_area, n_pixels, blur_pixels, exposure)
save_figure(folds_fig, "specular_caustics_folds")
folds_fig

Coral strokes mark the forward images of the zero-determinant contours at the initial frame, outlining the sharp boundaries where folded sheets of refracted rays overlap on the receiver.

## Multi-branch paths to a single receiver target

Inside a folded region, three distinct surface points refract light onto the same point on the receiver. Starting from three horizontal seeds beneath a single sinusoidal crest, Newton's method solves `derivative(ripple_landing)(trial_positions).solve(ripple_landing(trial_positions) - target)` to converge onto all three branches simultaneously.

In [ ]:
ripple_wavenumber = scenarios.ripple_wavenumber
ripple_amplitude = scenarios.ripple_amplitude
ripple_phase = scenarios.ripple_phase
seed_offsets = scenarios.seed_offsets
section_halfwidth = scenarios.section_halfwidth
incident_height = scenarios.incident_height
n_section = scenarios.n_section
n_fan = scenarios.n_fan
n_iterations = scenarios.n_iterations

In [ ]:
ripple_wavevector = mv.x * ripple_wavenumber                                      # [1] Wavevector+
ripple = partial(refracted_rays, wavevectors=ripple_wavevector, amplitudes=ripple_amplitude,
                 phases=ripple_phase, index_ratio=index_ratio)
ripple_landing = partial(receiver_landing, wavevectors=ripple_wavevector, amplitudes=ripple_amplitude,
                         phases=ripple_phase, depth=depth, index_ratio=index_ratio)
target = 0 * x                                                                    # [] Horizontal-
trial_positions = (x * seed_offsets).cast(Horizontal)                             # [3] Horizontal-
for _ in range(n_iterations):
    # Invert the local horizontal Jacobian to pull receiver residuals back into water-surface steps.
    trial_positions = trial_positions - derivative(ripple_landing)(trial_positions).solve(
        ripple_landing(trial_positions) - target
    )                                                                             # [3] Horizontal-

section = x * np.linspace(-section_halfwidth, section_halfwidth, n_section)       # [n_section] Horizontal-
fan = x * np.linspace(-section_halfwidth, section_halfwidth, n_fan)               # [n_fan] Horizontal-
surface_profile, _, _ = ripple(section)                                           # [n_section] Direction-
fan_surface, _, _ = ripple(fan)                                                   # [n_fan] Direction-
path_surface, _, _ = ripple(trial_positions)                                      # [3] Direction-
paths_fig = render.paths_figure(surface_profile, fan_surface, ripple_landing(fan) - up * depth,
                                path_surface, target - up * depth, incident_height)
save_figure(paths_fig, "specular_caustics_paths")
paths_fig

Faint blue-grey rays show the full refracted fan beneath the crest, while the three highlighted orange rays enter on the left flank, crest, and right flank and meet at the rust target marker on the receiver; as a target crosses a fold curve, two of those branches merge and the Jacobian becomes singular.

In [ ]:
# checks
# The landing Jacobian changes sign across folded sheets, and all three Newton branches hit the target.
assert bool(area_ratios.min() < 0) and bool(area_ratios.max() > 0)
assert (ripple_landing(trial_positions).scalar_norm_squared().abs() < 1e-12).all()
crossings = mv.x & path_surface                                                   # [3] Scalar+
assert bool(crossings[0] < -0.2) and bool(crossings[1].abs() < 1e-9) and bool(crossings[2] > 0.2)